In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

# ==========================================
# 1. Initialization
# ==========================================

@njit
def initialize_lattice(L):
    lattice = np.empty((L, L), dtype=np.int8)

    for i in range(L):
        for j in range(L):
            lattice[i, j] = 1 if np.random.random() < 0.5 else -1

    return lattice


# ==========================================
# 2. Optimized Metropolis Sweep
# ==========================================

@njit
def _metropolis_sweep_fast(lattice, T, h, J, spin_sum):
    L = lattice.shape[0]

    for _ in range(L * L):
        x = np.random.randint(0, L)
        y = np.random.randint(0, L)

        s = lattice[x, y]

        neighbors = (
            lattice[(x + 1) % L, y]
            + lattice[(x - 1) % L, y]
            + lattice[x, (y + 1) % L]
            + lattice[x, (y - 1) % L]
        )

        dE = 2.0 * s * (J * neighbors + h)

        if dE <= 0.0 or np.random.random() < np.exp(-dE / T):
            lattice[x, y] = -s
            spin_sum -= 2 * s

    return spin_sum


# Optional: compatible standalone sweep function
@njit
def metropolis_sweep(lattice, T, h=0.0, J=1.0):
    _metropolis_sweep_fast(lattice, T, h, J, 0)
    return lattice


# ==========================================
# 3. External Magnetic Field
# ==========================================

@njit
def External_Field(t, h0, period):
    return h0 * np.sin(2.0 * np.pi * t / period)


# ==========================================
# 4. Compiled Simulation Engine
# ==========================================

@njit
def _simulation_core(lattice, random_start,
                     T, h0, period, n, J, seed):

    # Seed Numba's internal random generator
    np.random.seed(seed)

    L = lattice.shape[0]
    N = L * L

    if random_start:
        lattice = initialize_lattice(L)

    spin_sum = 0
    for i in range(L):
        for j in range(L):
            spin_sum += lattice[i, j]

    total_mcs = n * period

    h_values = np.empty(total_mcs, dtype=np.float64)
    m_values = np.empty(total_mcs, dtype=np.float64)

    for t in range(total_mcs):
        h = External_Field(t, h0, period)

        spin_sum = _metropolis_sweep_fast(
            lattice, T, h, J, spin_sum
        )

        h_values[t] = h
        m_values[t] = spin_sum / N

    return h_values, m_values


# ==========================================
# 5. User-Facing Simulator
# ==========================================

def simulator(L=40, T=2.0, h0=0.5, period=100,
              n=10, lattice=None, seed=None, J=1.0):

    if L < 2 or T <= 0 or period < 1 or n < 1:
        raise ValueError("Check L, T, period and n.")

    if seed is None:
        seed = int(
            np.random.default_rng().integers(0, 2**31)
        )

    if lattice is None:
        initial = np.empty((L, L), dtype=np.int8)
        random_start = True
    else:
        initial = np.asarray(lattice)

        if initial.shape != (L, L):
            raise ValueError("Initial lattice shape must be (L, L).")

        if not np.all((initial == 1) | (initial == -1)):
            raise ValueError("Spins must be +1 or -1.")

        initial = np.array(initial, dtype=np.int8, copy=True)
        random_start = False

    return _simulation_core(
        initial, random_start,
        T, h0, period, n, J, int(seed)
    )


In [ ]:

import numpy as np
from numba import njit

@njit
def _simulation_core_with_snapshots(
    lattice, random_start, T, h0, period, n, J, seed
):
    # Same RNG initialization as the original simulator
    np.random.seed(seed)

    L = lattice.shape[0]
    N = L * L

    if random_start:
        lattice = initialize_lattice(L)

    spin_sum = np.sum(lattice)

    total_mcs = n * period

    h_values = np.empty(total_mcs, dtype=np.float64)
    m_values = np.empty(total_mcs, dtype=np.float64)

    # NEW: Store the lattice after each completed MCS
    snapshots = np.empty(
        (total_mcs, L, L), dtype=np.int8
    )

    for t in range(total_mcs):

        # Original external field function
        h = External_Field(t, h0, period)

        # ORIGINAL Metropolis update core
        spin_sum = _metropolis_sweep_fast(
            lattice, T, h, J, spin_sum
        )

        h_values[t] = h
        m_values[t] = spin_sum / N

        # NEW: Recording only, no extra random updates
        snapshots[t, :, :] = lattice

    return h_values, m_values, snapshots


def simulator_with_snapshots(
    L=40, T=2.0, h0=0.5, period=125,
    n=3, lattice=None, seed=42, J=1.0
):
    if L < 2 or T <= 0 or period < 1 or n < 1:
        raise ValueError("Invalid simulation parameters.")

    if lattice is None:
        initial = np.empty((L, L), dtype=np.int8)
        random_start = True
    else:
        initial = np.asarray(lattice, dtype=np.int8).copy()
        if initial.shape != (L, L):
            raise ValueError("Invalid lattice shape.")
        if not np.all((initial == -1) | (initial == 1)):
            raise ValueError("Spins must be -1 or +1.")
        random_start = False

    return _simulation_core_with_snapshots(
        initial, random_start,
        T, h0, period, n, J, int(seed)
    )


# Fixed physical parameters
L = 40
T = 2.0
h0 = 0.5
P = 125
n_cycles = 3
seed = 42

h_values, m_values, snapshots = simulator_with_snapshots(
    L=L, T=T, h0=h0,
    period=P, n=n_cycles, seed=seed
)

# Scientific consistency check:
# The snapshots extension must not change original dynamics.
h_original, m_original = simulator(
    L=L, T=T, h0=h0,
    period=P, n=n_cycles, seed=seed
)

assert np.array_equal(h_values, h_original)
assert np.array_equal(m_values, m_original)
assert np.array_equal(
    snapshots.mean(axis=(1, 2)), m_values
)

print("Original simulator and snapshot recording agree.")
print("Snapshots shape:", snapshots.shape)


In [ ]:

%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

N = len(h_values)
time = np.arange(1, N + 1)

# Create the figure ONCE
with plt.ioff():
    fig, axes = plt.subplots(
        1, 3,
        figsize=(13, 4),
        gridspec_kw={"width_ratios": [1, 1.4, 1]}
    )

ax_spin, ax_time, ax_loop = axes

# ------------------------------------
# Panel 1: Spin Configuration
# ------------------------------------
spin_image = ax_spin.imshow(
    snapshots[0],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    interpolation="nearest"
)

ax_spin.set_title("Spin Configuration")
ax_spin.set_xticks([])
ax_spin.set_yticks([])

# ------------------------------------
# Panel 2: h(t) and m(t)
# ------------------------------------
ax_time.plot(
    time, h_values,
    color="royalblue",
    label="h(t)"
)

ax_time.plot(
    time, m_values,
    color="darkorange",
    label="m(t)"
)

cursor = ax_time.axvline(
    1, color="black",
    linestyle="--", alpha=0.7
)

h_point, = ax_time.plot(
    [1], [h_values[0]],
    "o", color="royalblue"
)

m_point, = ax_time.plot(
    [1], [m_values[0]],
    "o", color="darkorange"
)

ax_time.set_xlim(1, N)
ax_time.set_ylim(-1.1, 1.1)
ax_time.set_xlabel("MCS")
ax_time.legend()

# ------------------------------------
# Panel 3: Hysteresis
# ------------------------------------
loop_line, = ax_loop.plot(
    [], [], color="purple", linewidth=2
)

loop_point, = ax_loop.plot(
    [], [], "ro", markersize=8
)

ax_loop.set_xlim(-1.1*h0, 1.1*h0)
ax_loop.set_ylim(-1.1, 1.1)
ax_loop.set_xlabel("External Field h")
ax_loop.set_ylabel("Magnetization m")
ax_loop.set_title("Hysteresis Evolution")
ax_loop.grid(alpha=0.2)

title = fig.suptitle("")

fig.tight_layout()

# ------------------------------------
# Interactive Controls
# ------------------------------------
slider = widgets.IntSlider(
    value=1,
    min=1,
    max=N,
    step=1,
    description="MCS",
    continuous_update=True,
    layout=widgets.Layout(width="650px")
)

play = widgets.Play(
    value=1,
    min=1,
    max=N,
    step=1,
    interval=100
)

widgets.jslink(
    (play, "value"),
    (slider, "value")
)

# ------------------------------------
# Fast frame update
# ------------------------------------
def update_frame(mcs):
    idx = mcs - 1

    # Update existing image
    spin_image.set_data(snapshots[idx])

    # Move time cursor
    cursor.set_xdata([mcs, mcs])

    h_point.set_data(
        [mcs], [h_values[idx]]
    )

    m_point.set_data(
        [mcs], [m_values[idx]]
    )

    # Update hysteresis trajectory
    loop_line.set_data(
        h_values[:idx+1],
        m_values[:idx+1]
    )

    loop_point.set_data(
        [h_values[idx]],
        [m_values[idx]]
    )

    title.set_text(
        f"MCS={mcs}  |  "
        f"h={h_values[idx]:+.3f}  "
        f"m={m_values[idx]:+.3f}"
    )

    # Redraw without rebuilding the figure
    fig.canvas.draw_idle()


def slider_changed(change):
    if change["name"] == "value":
        update_frame(change["new"])


slider.observe(slider_changed, names="value")

display(
    widgets.VBox([
        widgets.HBox([play, slider]),
        fig.canvas
    ])
)

update_frame(1)


In [ ]:
# ==========================================
# Demo: Temperature & External Field Response
# Animated Lattices + h(t) + m(t)
# ==========================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# ==========================================
# 1. Parameters
# ==========================================

L_demo = 40

temperatures_demo = [1.5, 2.0, 3.0]

h_initial_demo = 0.0
h_final_demo = -0.5

switch_time_demo = 45
n_sweeps_demo = 180
snapshot_interval_demo = 3

# ==========================================
# 2. Initialize Lattices
# ==========================================

lattices_demo = [
    np.ones((L_demo, L_demo), dtype=np.int8)
    for T in temperatures_demo
]

snapshots_demo = [
    [lattice.copy()] for lattice in lattices_demo
]

magnetizations_demo = [
    [np.mean(lattice)] for lattice in lattices_demo
]

times_demo = [0]
fields_demo = [h_initial_demo]

# ==========================================
# 3. Run Simulations
# ==========================================

for t in range(1, n_sweeps_demo + 1):

    # Apply negative field after switch time
    h = (
        h_initial_demo
        if t < switch_time_demo
        else h_final_demo
    )

    for i, T in enumerate(temperatures_demo):

        lattices_demo[i] = metropolis_sweep(
            lattices_demo[i],
            T,
            h=h
        )

    if t % snapshot_interval_demo == 0:

        times_demo.append(t)
        fields_demo.append(h)

        for i in range(len(temperatures_demo)):
            snapshots_demo[i].append(
                lattices_demo[i].copy()
            )

            magnetizations_demo[i].append(
                np.mean(lattices_demo[i])
            )

# ==========================================
# 4. Create Figure
# ==========================================

fig = plt.figure(figsize=(14, 8))

gs = fig.add_gridspec(
    2, 3,
    height_ratios=[1, 1],
    hspace=0.4,
    wspace=0.3
)

# Three lattice panels
lattice_axes = []
images = []

for i, T in enumerate(temperatures_demo):

    ax = fig.add_subplot(gs[0, i])

    im = ax.imshow(
        snapshots_demo[i][0],
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        interpolation="nearest"
    )

    ax.set_title(f"T = {T}")
    ax.set_xticks([])
    ax.set_yticks([])

    lattice_axes.append(ax)
    images.append(im)

# External field panel
ax_h = fig.add_subplot(gs[1, 0])

ax_h.set_xlim(0, n_sweeps_demo)
ax_h.set_ylim(
    min(h_initial_demo, h_final_demo) - 0.2,
    max(h_initial_demo, h_final_demo) + 0.2
)

ax_h.set_xlabel("MCS")
ax_h.set_ylabel("External Field h")
ax_h.set_title("External Field")
ax_h.grid(alpha=0.3)

h_line, = ax_h.step(
    [], [],
    where="post",
    color="black",
    linewidth=2
)

ax_h.axvline(
    switch_time_demo,
    linestyle="--",
    color="gray",
    alpha=0.5
)

# Magnetization panel
ax_m = fig.add_subplot(gs[1, 1:])

ax_m.set_xlim(0, n_sweeps_demo)
ax_m.set_ylim(-1.05, 1.05)

ax_m.set_xlabel("Monte Carlo Sweeps (MCS)")
ax_m.set_ylabel("Magnetization m")
ax_m.set_title("Magnetization Response")
ax_m.grid(alpha=0.3)

ax_m.axvline(
    switch_time_demo,
    linestyle="--",
    color="gray",
    alpha=0.5
)

m_lines = []

for T in temperatures_demo:
    line, = ax_m.plot(
        [], [],
        linewidth=2,
        label=f"T = {T}"
    )
    m_lines.append(line)

ax_m.legend()

title = fig.suptitle(
    "Temperature and External Field Response",
    fontsize=15
)

# ==========================================
# 5. Animation Update
# ==========================================

def update_demo(frame):

    current_time = times_demo[frame]
    current_h = fields_demo[frame]

    # Update the three lattices
    for i, T in enumerate(temperatures_demo):

        images[i].set_data(
            snapshots_demo[i][frame]
        )

        current_m = magnetizations_demo[i][frame]

        lattice_axes[i].set_title(
            f"T = {T}\n"
            f"m = {current_m:.3f}"
        )

        # Update corresponding m(t)
        m_lines[i].set_data(
            times_demo[:frame + 1],
            magnetizations_demo[i][:frame + 1]
        )

    # Update external field
    h_line.set_data(
        times_demo[:frame + 1],
        fields_demo[:frame + 1]
    )

    title.set_text(
        f"MCS = {current_time} | "
        f"External Field h = {current_h:.2f}"
    )

    return images + m_lines + [h_line, title]


# ==========================================
# 6. Display Animation
# ==========================================

ani_demo = FuncAnimation(
    fig,
    update_demo,
    frames=len(times_demo),
    interval=120,
    blit=False,
    repeat=True
)

plt.close(fig)

display(HTML(ani_demo.to_jshtml()))

In [ ]:
# ==========================================================
# Multi-Cycle Hysteresis Animation
# Simple Version + Selectable Driving Period
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output


# ==========================================================
# 1. Parameters
# ==========================================================

L_multi_demo = 40
T_multi_demo = 2.0
h0_multi_demo = 0.5

periods_multi_demo = [10, 50, 125, 300]

n_warmup_cycles = 15
n_show_cycles = 3

seed_multi_demo = 101

# Maximum animation frames per cycle
frames_per_cycle = 20


# ==========================================================
# 2. Generate Data for Selected Period
# ==========================================================

def generate_multi_cycle_data(P):

    n_total_cycles = n_warmup_cycles + n_show_cycles

    h_all, m_all = simulator(
        L=L_multi_demo,
        T=T_multi_demo,
        h0=h0_multi_demo,
        period=P,
        n=n_total_cycles,
        seed=seed_multi_demo
    )

    n_points = n_show_cycles * P

    # Select last three complete cycles
    h_show = np.asarray(h_all[-n_points:])
    m_show = np.asarray(m_all[-n_points:])

    t_show = np.arange(n_points)

    return t_show, h_show, m_show


# ==========================================================
# 3. Build Animation
# ==========================================================

def build_multi_cycle_animation(P):

    t, h, m = generate_multi_cycle_data(P)

    n_points = len(t)

    # ------------------------------------------------------
    # Figure
    # ------------------------------------------------------

    fig, (ax_time, ax_loop) = plt.subplots(
        1, 2,
        figsize=(12, 5)
    )

    # ------------------------------------------------------
    # Left: h(t) and m(t)
    # ------------------------------------------------------

    h_line, = ax_time.plot(
        [], [],
        '--',
        color='darkorange',
        linewidth=2,
        label='External Field h'
    )

    m_line, = ax_time.plot(
        [], [],
        color='steelblue',
        linewidth=2,
        label='Magnetization m'
    )

    time_marker = ax_time.axvline(
        0,
        color='black',
        alpha=0.45,
        linestyle='--'
    )

    ax_time.set_xlim(0, n_points)
    ax_time.set_ylim(-1.1, 1.1)

    ax_time.set_xlabel('MCS')
    ax_time.set_ylabel('h(t), m(t)')
    ax_time.set_title('External Field and Magnetization')

    ax_time.grid(alpha=0.25)
    ax_time.legend(loc='lower left')

    # Mark the boundaries between cycles
    for c in range(1, n_show_cycles):

        ax_time.axvline(
            c * P,
            color='gray',
            linestyle=':',
            alpha=0.5
        )

    # ------------------------------------------------------
    # Right: Hysteresis Loops
    # ------------------------------------------------------

    colors = plt.cm.tab10(
        np.arange(n_show_cycles)
    )

    loop_lines = []

    for c in range(n_show_cycles):

        line, = ax_loop.plot(
            [], [],
            color=colors[c],
            linewidth=2,
            label=f'Cycle {c + 1}'
        )

        loop_lines.append(line)

    loop_point, = ax_loop.plot(
        [], [],
        'ko',
        markersize=6
    )

    ax_loop.set_xlim(
        -1.2 * h0_multi_demo,
        1.2 * h0_multi_demo
    )

    ax_loop.set_ylim(-1.05, 1.05)

    ax_loop.set_xlabel('External Field h')
    ax_loop.set_ylabel('Magnetization m')
    ax_loop.set_title('Hysteresis Loops')

    ax_loop.grid(alpha=0.25)
    ax_loop.legend(loc='lower right')

    title = fig.suptitle(
        f'T = {T_multi_demo} | P = {P} | L = {L_multi_demo}'
    )

    plt.tight_layout()

    # ======================================================
    # 4. Animation Update
    # ======================================================

    def update(frame):

        end = frame + 1

        # ---- Time series ----

        h_line.set_data(
            t[:end],
            h[:end]
        )

        m_line.set_data(
            t[:end],
            m[:end]
        )

        time_marker.set_xdata(
            [frame, frame]
        )

        # ---- Hysteresis loops ----

        for c in range(n_show_cycles):

            start = c * P
            stop = min(end, (c + 1) * P)

            if stop > start:

                loop_lines[c].set_data(
                    h[start:stop],
                    m[start:stop]
                )

            else:

                loop_lines[c].set_data([], [])

        # ---- Current point ----

        loop_point.set_data(
            [h[frame]],
            [m[frame]]
        )

        current_cycle = frame // P + 1

        title.set_text(
            f'T = {T_multi_demo} | '
            f'P = {P} | '
            f'Cycle = {current_cycle}/{n_show_cycles} | '
            f'MCS = {frame}'
        )

        return (
            h_line,
            m_line,
            time_marker,
            loop_point,
            *loop_lines
        )

    # ======================================================
    # 5. Select Animation Frames
    # ======================================================

    frames = []

    for c in range(n_show_cycles):

        start = c * P
        end = (c + 1) * P - 1

        cycle_frames = np.linspace(
            start,
            end,
            num=min(P, frames_per_cycle),
            dtype=int
        )

        frames.extend(cycle_frames)

    frames = np.unique(frames)

    # ======================================================
    # 6. Create Animation
    # ======================================================

    ani = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=120,
        blit=False,
        repeat=True
    )

    plt.close(fig)

    return HTML(
        ani.to_jshtml(
            fps=8,
            default_mode='loop'
        )
    )


# ==========================================================
# 7. Period Selector
# ==========================================================

period_selector_multi = widgets.Dropdown(
    options=periods_multi_demo,
    value=125,
    description='Period P:',
    layout=widgets.Layout(width='220px')
)

animation_output_multi = widgets.Output()


def update_selected_period(change=None):

    P = int(period_selector_multi.value)

    with animation_output_multi:

        clear_output(wait=True)

        display(
            build_multi_cycle_animation(P)
        )


period_selector_multi.observe(
    update_selected_period,
    names='value'
)

# ==========================================================
# 8. Display
# ==========================================================

display(
    widgets.VBox([
        period_selector_multi,
        animation_output_multi
    ])
)

update_selected_period()

In [ ]:
def loop_area(h, m):
    h = np.asarray(h)
    m = np.asarray(m)

    dh = np.roll(h, -1) - h
    m_avg = (m + np.roll(m, -1)) / 2

    return abs(np.sum(m_avg * dh))
periods = [10, 20, 50, 75, 100, 125, 150, 200, 300]
seeds = [101, 202, 303, 404, 505]

L = 40
T = 2.0
h0 = 0.5

n_cycles = 30
burn_in = 10

# Store results: rows = seeds, columns = periods
all_areas = np.zeros((len(seeds), len(periods)))

# Run simulations
for i, seed in enumerate(seeds):
    for j, P in enumerate(periods):

        h, m = simulator(
            L=L, T=T, h0=h0,
            period=P, n=n_cycles, seed=seed
        )

        cycle_areas = []

        for k in range(burn_in, n_cycles):
            start = k * P
            end = (k + 1) * P

            A = loop_area(h[start:end], m[start:end])
            cycle_areas.append(A)

        all_areas[i, j] = np.mean(cycle_areas)

# Ensemble statistics
mean_areas = np.mean(all_areas, axis=0)

area_errors = (
    np.std(all_areas, axis=0, ddof=1)
    / np.sqrt(len(seeds))
)

# Plot
plt.figure(figsize=(9, 6))

# Individual seed results
for i, seed in enumerate(seeds):
    plt.scatter(
        periods, all_areas[i],
        alpha=0.45, s=30,
        label=f"Seed {seed}"
    )

# Ensemble mean + error bars
plt.errorbar(
    periods, mean_areas,
    yerr=area_errors,
    fmt='ko-', capsize=4,
    linewidth=1.5,
    label="Mean ± SEM"
)

plt.xlabel("Driving Period P (MCS)")
plt.ylabel("Hysteresis Loop Area A")
plt.title("Hysteresis Area vs Driving Period")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
periods_s = [10, 125, 300] #some certain periods that we are interested in

plt.figure(figsize=(8, 6))

for P in periods_s:
    h, m = simulator(
        L=40, T=2.0, h0=0.5,
        period=P, n=30, seed=101
    )

    plt.plot(h[-P:], m[-P:], label=f"P = {P}")

plt.xlabel("External Field h")
plt.ylabel("Magnetization m")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# ==================================================
# PERIOD SWEEP EXPLORER — High Quality Version
# ==================================================

import numpy as np
import time

L_sweep_demo = 40
T_sweep_demo = 2.0
h0_sweep_demo = 0.5

# Denser sampling of P
periods_sweep_demo = list(np.unique(np.r_[
    np.arange(10, 60, 5),
    np.arange(60, 201, 10),
    np.arange(220, 321, 20)
]))

seeds_sweep_demo = [101, 202, 303]

n_cycles_sweep_demo = 24
burn_in_sweep_demo = 12

loop_data_sweep = {}
mean_areas_sweep = []

start_time = time.perf_counter()

for P in periods_sweep_demo:

    # Collect phase-aligned cycles from all seeds
    all_cycles = []

    for seed in seeds_sweep_demo:

        h, m = simulator(
            L=L_sweep_demo,
            T=T_sweep_demo,
            h0=h0_sweep_demo,
            period=int(P),
            n=n_cycles_sweep_demo,
            seed=seed
        )

        h = np.asarray(h)
        m = np.asarray(m)

        # Shape: (number of cycles, P)
        m_cycles = m.reshape(n_cycles_sweep_demo, P)

        # Remove transient cycles
        steady_cycles = m_cycles[burn_in_sweep_demo:]

        all_cycles.append(steady_cycles)

    # Combine all seeds and cycles
    all_cycles = np.concatenate(all_cycles, axis=0)

    # Average magnetization at each driving phase
    m_mean = np.mean(all_cycles, axis=0)

    # The field is identical for every cycle
    h_cycle = h[-P:].copy()

    # Area of the phase-averaged loop
    dh = np.roll(h_cycle, -1) - h_cycle
    m_mid = (m_mean + np.roll(m_mean, -1)) / 2

    W_mean = -np.sum(m_mid * dh)
    A_mean = abs(W_mean)

    loop_data_sweep[P] = {
        "h": h_cycle,
        "m": m_mean,
        "raw_m": all_cycles[-1].copy(),
        "A_mean": A_mean,
        "W_mean": W_mean
    }

    mean_areas_sweep.append(A_mean)

    print(
        f"P = {P:3d} | "
        f"A = {A_mean:.4f}",
        flush=True
    )

mean_areas_sweep = np.asarray(mean_areas_sweep)

print(
    f"\nCompleted in "
    f"{time.perf_counter() - start_time:.2f} seconds"
)

In [ ]:
# ==================================================
# PERIOD SWEEP EXPLORER — Part 2
# Interactive Loop Morphology + A(P)
# ==================================================

import ipywidgets as widgets
from IPython.display import display
from io import BytesIO

# ==================================================
# 1. Render one image per Period
# ==================================================

rendered_sweep_images = {}

for selected_P in periods_sweep_demo:

    data = loop_data_sweep[selected_P]

    fig, axes = plt.subplots(
        1, 2,
        figsize=(10, 4.5)
    )

    ax_loop, ax_area = axes

    # ----------------------------------------------
    # Left: Hysteresis Loop
    # ----------------------------------------------

    ax_loop.plot(
        data["h"],
        data["m"],
        color="teal",
        linewidth=2
    )

    # Mark starting and ending points
    ax_loop.scatter(
        data["h"][0],
        data["m"][0],
        color="green",
        marker="o",
        s=50,
        label="Start"
    )

    ax_loop.scatter(
        data["h"][-1],
        data["m"][-1],
        color="red",
        marker="x",
        s=50,
        label="End"
    )

    ax_loop.set_xlim(
        -1.2 * h0_sweep_demo,
        1.2 * h0_sweep_demo
    )

    ax_loop.set_ylim(-1.05, 1.05)

    ax_loop.set_xlabel("External Field h")
    ax_loop.set_ylabel("Magnetization m")
    ax_loop.set_title(f"Hysteresis Loop | P = {selected_P}")
    ax_loop.grid(alpha=0.3)
    ax_loop.legend(loc="lower right")

    # ----------------------------------------------
    # Right: Area vs Period
    # ----------------------------------------------

    ax_area.plot(
        periods_sweep_demo,
        mean_areas_sweep,
        "o-",
        color="darkorange",
        linewidth=2,
        markersize=5
    )

    # Highlight selected period
    selected_area = data["A_mean"]

    ax_area.scatter(
        [selected_P],
        [selected_area],
        color="red",
        s=110,
        zorder=5
    )

    ax_area.axvline(
        selected_P,
        color="red",
        linestyle="--",
        alpha=0.4
    )

    ax_area.set_xlim(0, max(periods_sweep_demo) + 20)
    ax_area.set_ylim(
        0,
        max(2.0, np.max(mean_areas_sweep) * 1.15)
    )

    ax_area.set_xlabel("Driving Period P (MCS)")
    ax_area.set_ylabel("Mean Hysteresis Area")
    ax_area.set_title("Area vs Driving Period")
    ax_area.grid(alpha=0.3)

    fig.suptitle(
        f"T = {T_sweep_demo} | "
        f"L = {L_sweep_demo} | "
        f"P = {selected_P} | "
        f"Mean A = {selected_area:.3f}",
        fontsize=13
    )

    fig.tight_layout()

    # Save figure into memory
    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=110
    )

    rendered_sweep_images[selected_P] = buffer.getvalue()

    buffer.close()
    plt.close(fig)

# ==================================================
# 2. Create Interactive Controls
# ==================================================

period_slider_sweep = widgets.SelectionSlider(
    options=[
        (f"P = {P}", P)
        for P in periods_sweep_demo
    ],
    value=10,
    description="Period:",
    continuous_update=False,
    layout=widgets.Layout(width="650px")
)

# Pre-rendered image
sweep_image = widgets.Image(
    value=rendered_sweep_images[10],
    format="png",
    layout=widgets.Layout(width="100%")
)

# ==================================================
# 3. Update Display — No Recalculation
# ==================================================

def update_period_sweep(change):

    P = change["new"]

    # Only replace image bytes
    sweep_image.value = rendered_sweep_images[P]


period_slider_sweep.observe(
    update_period_sweep,
    names="value"
)

# ==================================================
# 4. Optional Auto-Play Control
# ==================================================

play_sweep = widgets.Play(
    value=0,
    min=0,
    max=len(periods_sweep_demo) - 1,
    step=1,
    interval=1200
)

# Play controls the selected Period index
widgets.jslink(
    (play_sweep, "value"),
    (period_slider_sweep, "index")
)

# ==================================================
# 5. Display
# ==================================================

display(
    widgets.VBox([
        widgets.HBox([
            play_sweep,
            period_slider_sweep
        ]),
        sweep_image
    ])
)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Experimental Parameters
# ==========================================

Q_L = 40
Q_T = 2.0
Q_h0 = 0.5

Q_periods = [10, 20, 50, 75, 100, 125, 150, 200, 300]
Q_seeds = [101, 202, 303, 404, 505]

Q_n_cycles = 60
Q_burn_in = 20

assert Q_n_cycles > Q_burn_in
assert len(Q_seeds) >= 2


# ==========================================
# 2. Calculate Q for Each Cycle
# ==========================================

def calculate_Q(m, period):
    """
    Q_k = mean magnetization over cycle k.
    One magnetization sample per completed MCS.
    """
    m = np.asarray(m, dtype=float)

    if len(m) % period != 0:
        raise ValueError("Incomplete driving cycles.")

    return m.reshape(-1, period).mean(axis=1)


# ==========================================
# 3. Run Existing Simulator
# ==========================================

# Shape: (seed, period)
Q_all_runs = np.zeros((
    len(Q_seeds), len(Q_periods)
))

# Preserve signed Q_k for diagnostics
Q_cycle_data = {}

for i, seed in enumerate(Q_seeds):
    for j, P in enumerate(Q_periods):

        h, m = simulator(
            L=Q_L,
            T=Q_T,
            h0=Q_h0,
            period=P,
            n=Q_n_cycles,
            seed=seed
        )

        # Signed Q for every cycle
        Q_cycles = calculate_Q(m, P)

        Q_cycle_data[(seed, P)] = Q_cycles.copy()

        # Discard initial transient cycles
        Q_retained = Q_cycles[Q_burn_in:]

        # IMPORTANT: absolute value BEFORE averaging
        Q_all_runs[i, j] = np.mean(np.abs(Q_retained))


# ==========================================
# 4. Ensemble Statistics
# ==========================================

Q_mean = np.mean(Q_all_runs, axis=0)

# SEM calculated across independent seeds
Q_sem = (
    np.std(Q_all_runs, axis=0, ddof=1)
    / np.sqrt(len(Q_seeds))
)




In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Experimental Parameters
# ==========================================

Q_L = 40
Q_T = 2.0
Q_h0 = 0.5

Q_periods = [10, 20, 50, 75, 100, 125, 150, 200, 300]
Q_seeds = [101, 202, 303, 404, 505]

Q_n_cycles = 60
Q_burn_in = 20

assert Q_n_cycles > Q_burn_in
assert len(Q_seeds) >= 2


# ==========================================
# 2. Calculate Q for Each Cycle
# ==========================================

def calculate_Q(m, period):
    """
    Q_k = mean magnetization over cycle k.
    One magnetization sample per completed MCS.
    """
    m = np.asarray(m, dtype=float)

    if len(m) % period != 0:
        raise ValueError("Incomplete driving cycles.")

    return m.reshape(-1, period).mean(axis=1)


# ==========================================
# 3. Run Existing Simulator
# ==========================================

# Shape: (seed, period)
Q_all_runs = np.zeros((
    len(Q_seeds), len(Q_periods)
))

# Preserve signed Q_k for diagnostics
Q_cycle_data = {}

for i, seed in enumerate(Q_seeds):
    for j, P in enumerate(Q_periods):

        h, m = simulator(
            L=Q_L,
            T=Q_T,
            h0=Q_h0,
            period=P,
            n=Q_n_cycles,
            seed=seed
        )

        # Signed Q for every cycle
        Q_cycles = calculate_Q(m, P)

        Q_cycle_data[(seed, P)] = Q_cycles.copy()

        # Discard initial transient cycles
        Q_retained = Q_cycles[Q_burn_in:]

        # IMPORTANT: absolute value BEFORE averaging
        Q_all_runs[i, j] = np.mean(np.abs(Q_retained))


# ==========================================
# 4. Ensemble Statistics
# ==========================================

Q_mean = np.mean(Q_all_runs, axis=0)

# SEM calculated across independent seeds
Q_sem = (
    np.std(Q_all_runs, axis=0, ddof=1)
    / np.sqrt(len(Q_seeds))
)


In [ ]:

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# Figure 1: Q(P) — Main Research Result
# ==========================================

fig, ax = plt.subplots(figsize=(8, 5))

for i, seed in enumerate(Q_seeds):
    ax.scatter(
        Q_periods,
        Q_all_runs[i],
        alpha=0.4,
        s=35,
        label=f"Seed {seed}"
    )

ax.errorbar(
    Q_periods,
    Q_mean,
    yerr=Q_sem,
    fmt="ko-",
    capsize=4,
    linewidth=1.5,
    markersize=5,
    label="Mean ± SEM"
)

ax.set_xlabel("Driving Period P (MCS)")
ax.set_ylabel("Mean |Q|")
ax.set_title(
    f"Dynamic Order Parameter | "
    f"L={Q_L}, T={Q_T}, h0={Q_h0}"
)

ax.set_ylim(0, 1.05)
ax.grid(alpha=0.25)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


# ==========================================
# Figure 2: Q per Cycle — Diagnostic
# ==========================================

fig, ax = plt.subplots(figsize=(8, 4.5))

diagnostic_seed = Q_seeds[0]

for P in [10, 125, 300]:
    Q_cycles = Q_cycle_data[(diagnostic_seed, P)]

    ax.plot(
        np.arange(1, len(Q_cycles) + 1),
        Q_cycles,
        marker=".",
        linewidth=1.4,
        label=f"P = {P}"
    )

ax.axvline(
    Q_burn_in + 0.5,
    color="gray",
    linestyle="--",
    label="Burn-in cutoff"
)

ax.axhline(0, color="black", linewidth=0.8)

ax.set_xlabel("Driving Cycle")
ax.set_ylabel("Signed Q per Cycle")
ax.set_title(f"Cycle-by-Cycle Q | Seed {diagnostic_seed}")
ax.set_ylim(-1.05, 1.05)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

L = 40
seeds = [101, 202, 303, 404, 505, 606, 707, 808]

m0_values = []

for seed in seeds:
    lattice = make_initial_state("random", L, seed)
    m0 = np.mean(lattice)
    m0_values.append(m0)
    print(f"Seed {seed}: initial magnetization = {m0:+.5f}")

print("\nMean initial magnetization:", np.mean(m0_values))
print("Expected standard deviation:", 1 / L)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ==================================================
# Fast Driving: Initial-State Memory Test
# Uses the EXISTING simulator()
# ==================================================

# 1. Independent experimental parameters
MEM_L = 40
MEM_T = 2.0
MEM_h0 = 0.5

MEM_periods = [10, 20, 50]
MEM_seeds = [101, 202, 303, 404, 505, 606, 707, 808]
MEM_states = ["all-up", "all-down", "random"]

# Equal physical simulation duration for every P
MEM_total_mcs = 4000
MEM_burn_mcs = 2000

results = {}

# ==================================================
# 2. Generate controlled initial configurations
# ==================================================

def make_initial_state(kind, L, seed):
    if kind == "all-up":
        return np.ones((L, L), dtype=np.int8)

    if kind == "all-down":
        return -np.ones((L, L), dtype=np.int8)

    if kind == "random":
        rng = np.random.default_rng(100000 + seed)
        return rng.choice(
            np.array([-1, 1], dtype=np.int8),
            size=(L, L)
        )

    raise ValueError("Unknown initial state")


# ==================================================
# 3. Run the original simulator
# ==================================================

for P in MEM_periods:

    assert MEM_total_mcs % P == 0
    assert MEM_burn_mcs % P == 0

    n_cycles = MEM_total_mcs // P
    burn_cycles = MEM_burn_mcs // P

    for kind in MEM_states:
        for seed in MEM_seeds:

            initial = make_initial_state(
                kind, MEM_L, seed
            )

            initial_m = initial.mean()

            h, m = simulator(
                L=MEM_L,
                T=MEM_T,
                h0=MEM_h0,
                period=P,
                n=n_cycles,
                lattice=initial,
                seed=seed
            )

            assert len(m) == MEM_total_mcs

            # Each row is one driving cycle
            m_cycles = np.asarray(m).reshape(
                n_cycles, P
            )

            # Signed Q for each cycle
            Q_cycles = m_cycles.mean(axis=1)

            Q_late = Q_cycles[burn_cycles:]
            m_late = m_cycles[burn_cycles:]

            # Count cycles with a substantial reversal
            # This is a diagnostic, not a new order parameter
            reversal = (
                (m_late.min(axis=1) < -0.4)
                & (m_late.max(axis=1) > 0.4)
            )

            results[(P, kind, seed)] = {
                "m0": initial_m,
                "Q_cycles": Q_cycles.copy(),
                "late_signed_Q": Q_late.mean(),
                "late_abs_Q": np.abs(Q_late).mean(),
                "reversal_fraction": reversal.mean()
            }

    print(f"P = {P} completed")


# ==================================================
# 4. Figure 1: Q evolution for different initial states
# ==================================================

fig, axes = plt.subplots(
    1, len(MEM_periods),
    figsize=(15, 4.5),
    sharex=True,
    sharey=True
)

example_seed = MEM_seeds[0]

for ax, P in zip(axes, MEM_periods):

    for kind in MEM_states:

        Q = results[(P, kind, example_seed)]["Q_cycles"]

        # Midpoint of each cycle in MCS
        t = (np.arange(len(Q)) + 0.5) * P

        ax.plot(
            t, Q,
            linewidth=1.5,
            label=kind
        )

    ax.axvline(
        MEM_burn_mcs,
        linestyle="--",
        color="gray",
        label="Burn-in"
    )

    ax.axhline(0, color="black", linewidth=0.8)

    ax.set_title(f"P = {P} MCS")
    ax.set_xlabel("Elapsed MCS")
    ax.set_ylim(-1.05, 1.05)
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Signed Q per Cycle")
axes[-1].legend(fontsize=8)

fig.suptitle(
    f"Initial-State Memory | L={MEM_L}, "
    f"T={MEM_T}, h0={MEM_h0}, Seed={example_seed}"
)

plt.tight_layout()
plt.show()


# ==================================================
# 5. Figure 2: Late-time signed Q across seeds
# ==================================================

fig, ax = plt.subplots(figsize=(10, 5))

offsets = {
    "all-up": -0.22,
    "all-down": 0.0,
    "random": 0.22
}

for kind in MEM_states:

    x_positions = []
    means = []
    sems = []

    for j, P in enumerate(MEM_periods):

        values = np.array([
            results[(P, kind, seed)]["late_signed_Q"]
            for seed in MEM_seeds
        ])

        x = j + offsets[kind]

        # Individual independent runs
        ax.scatter(
            np.full(len(values), x),
            values,
            alpha=0.4,
            s=28
        )

        x_positions.append(x)
        means.append(values.mean())
        sems.append(
            values.std(ddof=1) / np.sqrt(len(values))
        )

    ax.errorbar(
        x_positions,
        means,
        yerr=sems,
        fmt="o-",
        capsize=4,
        linewidth=1.5,
        label=kind
    )

ax.axhline(0, color="gray", linestyle="--")
ax.set_xticks(range(len(MEM_periods)))
ax.set_xticklabels([f"P={P}" for P in MEM_periods])
ax.set_ylabel("Late-Time Mean Signed Q")
ax.set_title("Initial-State Dependence Across Seeds")
ax.set_ylim(-1.05, 1.05)
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()


# ==================================================
# 6. Numerical summary
# ==================================================

print("\nFAST DRIVING INITIAL-STATE TEST")
print(f"Total duration = {MEM_total_mcs} MCS")
print(f"Discarded = {MEM_burn_mcs} MCS")
print(f"Independent seeds = {len(MEM_seeds)}")
print("-" * 75)

for P in MEM_periods:
    print(f"\nP = {P}")

    for kind in MEM_states:

        signed = np.array([
            results[(P, kind, seed)]["late_signed_Q"]
            for seed in MEM_seeds
        ])

        abs_q = np.array([
            results[(P, kind, seed)]["late_abs_Q"]
            for seed in MEM_seeds
        ])

        reversal = np.array([
            results[(P, kind, seed)]["reversal_fraction"]
            for seed in MEM_seeds
        ])

        print(
            f"{kind:9s} | "
            f"signed Q = {signed.mean():+.3f} | "
            f"mean |Q| = {abs_q.mean():.3f} | "
            f"reversal fraction = {reversal.mean():.3f}"
        )


In [ ]:
for P in [10, 20, 50]:
    print(f"\nP = {P}")

    for seed in MEM_seeds:
        r = results[(P, "random", seed)]

        print(
            f"Seed {seed}: "
            f"m0 = {r['m0']:+.4f}, "
            f"Q_late = {r['late_signed_Q']:+.4f}"
        )

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# TEMPERATURE DEPENDENCE OF DYNAMIC ORDER
# Fixed P = 10 | Original Simulator
# =====================================================

# 1. Independent parameters

QT_L = 40
QT_P = 10
QT_h0 = 0.5

QT_temperatures = [1.5, 1.8, 2.0, 2.27, 2.5, 3.2]
QT_seeds = [101, 202, 303, 404, 505, 606, 707, 808]

QT_total_mcs = 4000
QT_burn_mcs = 2000

assert QT_total_mcs % QT_P == 0
assert QT_burn_mcs % QT_P == 0

QT_n_cycles = QT_total_mcs // QT_P
QT_burn_cycles = QT_burn_mcs // QT_P

# Shape: (temperature, seed, driving cycle)
QT_cycle_data = np.empty((
    len(QT_temperatures),
    len(QT_seeds),
    QT_n_cycles
))

# Shape: (temperature, seed)
QT_run_means = np.empty((
    len(QT_temperatures),
    len(QT_seeds)
))


# =====================================================
# 2. Run existing simulator
# =====================================================

for i, T in enumerate(QT_temperatures):

    for j, seed in enumerate(QT_seeds):

        # Same random initial configuration
        # for the same seed at different temperatures
        rng = np.random.default_rng(100000 + seed)

        initial = rng.choice(
            np.array([-1, 1], dtype=np.int8),
            size=(QT_L, QT_L)
        )

        # ORIGINAL simulator — no new Metropolis core
        h, m = simulator(
            L=QT_L,
            T=T,
            h0=QT_h0,
            period=QT_P,
            n=QT_n_cycles,
            lattice=initial,
            seed=seed
        )

        assert len(m) == QT_total_mcs

        # Signed Q for every driving cycle
        Q_cycles = np.asarray(m).reshape(
            QT_n_cycles, QT_P
        ).mean(axis=1)

        QT_cycle_data[i, j, :] = Q_cycles

        # Absolute value before averaging cycles
        QT_run_means[i, j] = np.mean(
            np.abs(Q_cycles[QT_burn_cycles:])
        )

    print(f"T = {T:.2f} completed")


# =====================================================
# 3. Ensemble Statistics
# =====================================================

QT_mean = QT_run_means.mean(axis=1)

QT_sem = (
    QT_run_means.std(axis=1, ddof=1)
    / np.sqrt(len(QT_seeds))
)

# Mean absolute Q at each cycle across seeds
QT_abs_evolution = np.abs(QT_cycle_data).mean(axis=1)

QT_abs_evolution_sem = (
    np.abs(QT_cycle_data).std(axis=1, ddof=1)
    / np.sqrt(len(QT_seeds))
)

cycle_times = (
    np.arange(QT_n_cycles) + 0.5
) * QT_P


# =====================================================
# 4. Figure 1: Evolution of |Q| at Different T
# =====================================================

fig, ax = plt.subplots(figsize=(9, 5.5))

for i, T in enumerate(QT_temperatures):

    y = QT_abs_evolution[i]
    err = QT_abs_evolution_sem[i]

    line, = ax.plot(
        cycle_times, y,
        linewidth=1.6,
        label=f"T = {T}"
    )

    ax.fill_between(
        cycle_times,
        np.maximum(0, y - err),
        np.minimum(1, y + err),
        color=line.get_color(),
        alpha=0.10
    )

ax.axvline(
    QT_burn_mcs,
    color="black",
    linestyle="--",
    alpha=0.7,
    label="Burn-in cutoff"
)

ax.set_xlabel("Elapsed Monte Carlo Sweeps (MCS)")
ax.set_ylabel("Ensemble Mean |Q| per Cycle")

ax.set_title(
    f"Evolution of Dynamic Order | "
    f"P={QT_P}, L={QT_L}, h0={QT_h0}"
)

ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


# =====================================================
# 5. Figure 2: Long-Time |Q| vs Temperature
# =====================================================

fig, ax = plt.subplots(figsize=(8, 5))

for j, seed in enumerate(QT_seeds):

    ax.scatter(
        QT_temperatures,
        QT_run_means[:, j],
        alpha=0.35,
        s=30,
        label=f"Seed {seed}"
    )

ax.errorbar(
    QT_temperatures,
    QT_mean,
    yerr=QT_sem,
    fmt="ko-",
    capsize=4,
    linewidth=1.5,
    label="Mean ± SEM"
)

# Equilibrium zero-field Tc as reference only
ax.axvline(
    2.269,
    color="gray",
    linestyle="--",
    alpha=0.65,
    label="Zero-field Tc"
)

ax.set_xlabel("Temperature T")
ax.set_ylabel("Mean |Q|")

ax.set_title(
    f"Temperature Dependence of Dynamic Order | "
    f"P={QT_P}, L={QT_L}"
)

ax.set_ylim(0, 1.05)
ax.grid(alpha=0.25)
ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()


# =====================================================
# 6. Numerical Summary
# =====================================================

print("\nTemperature Dependence of Dynamic Order")
print(f"P = {QT_P}, L = {QT_L}, h0 = {QT_h0}")
print(f"Burn-in = {QT_burn_mcs} MCS")
print("-" * 45)

for T, mean, sem in zip(
    QT_temperatures, QT_mean, QT_sem
):
    print(
        f"T = {T:4.2f} | "
        f"<|Q|> = {mean:.4f} ± {sem:.4f}"
    )


In [ ]:

# ==========================================
# Figure 3 (Revised):
# Zero Field vs Fast Driving at ALL temperatures
# ==========================================

fig, axes = plt.subplots(
    2, 3,
    figsize=(15, 9),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

time = (
    np.arange(ZF_n_cycles) + 0.5
) * ZF_P

conditions = [
    (Q_driven, "Fast driving, h0 = 0.5"),
    (Q_zero, "Zero field, h0 = 0")
]

for i, (T, ax) in enumerate(zip(ZF_T, axes)):

    for data, label in conditions:

        # Shape: (independent seeds, cycles)
        values = np.abs(data[i])

        mean = values.mean(axis=0)
        sem = (
            values.std(axis=0, ddof=1)
            / np.sqrt(len(ZF_seeds))
        )

        line, = ax.plot(
            time,
            mean,
            linewidth=1.7,
            label=label
        )

        ax.fill_between(
            time,
            np.maximum(0, mean - sem),
            np.minimum(1, mean + sem),
            color=line.get_color(),
            alpha=0.15
        )

    ax.axvline(
        ZF_burn_cycles * ZF_P,
        color="gray",
        linestyle="--",
        alpha=0.7
    )

    ax.set_title(f"T = {T:.2f}")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)

    if i >= 3:
        ax.set_xlabel("Elapsed MCS")

    if i % 3 == 0:
        ax.set_ylabel("Ensemble Mean |Q| per Window")

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 0.965)
)

fig.suptitle(
    f"Zero-Field Control vs Fast Driving "
    f"| L={ZF_L}, P={ZF_P}",
    fontsize=15,
    y=1.01
)

plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()



In [ ]:

import numpy as np

def loop_area(h, m):
    """
    Calculate the absolute hysteresis loop integral
    for one complete driving cycle.
    """
    h = np.asarray(h, dtype=float)
    m = np.asarray(m, dtype=float)

    assert len(h) == len(m)

    # Periodic closure
    dh = np.roll(h, -1) - h
    m_avg = (m + np.roll(m, -1)) / 2

    # Absolute loop integral
    A = np.abs(np.sum(m_avg * dh))

    return A


In [ ]:
temperatures_s = [1.5, 1.8, 2.0, 2.3]
periods_t = [10, 20, 50, 75, 100, 125, 150, 200, 300, 450]
seeds = [101, 202, 303, 404, 505]

L = 40
h0 = 0.5
n_cycles = 30
burn_in = 10

# Shape: (Temperature, Period, Seed)
all_areas = np.zeros((
    len(temperatures_s),
    len(periods_t),
    len(seeds)
))

for i, T in enumerate(temperatures_s):
    for j, P in enumerate(periods_t):
        for k, seed in enumerate(seeds):

            h, m = simulator(
                L=L, T=T, h0=h0,
                period=P, n=n_cycles,
                seed=seed
            )

            cycle_areas = []

            for cycle in range(burn_in, n_cycles):
                start = cycle * P
                end = (cycle + 1) * P

                A = loop_area(
                    h[start:end],
                    m[start:end]
                )

                cycle_areas.append(A)

            all_areas[i, j, k] = np.mean(cycle_areas)

mean_A = np.mean(all_areas, axis=2)
error_A = np.std(all_areas, axis=2, ddof=1) / np.sqrt(len(seeds))

plt.figure(figsize=(9, 6))

for i, T in enumerate(temperatures_s):
    plt.errorbar(
        periods_t,
        mean_A[i],
        yerr=error_A[i],
        fmt='o-',
        capsize=3,
        label=f'T = {T}'
    )

plt.xlabel("Driving Period P (MCS)")
plt.ylabel("Hysteresis Loop Area A")
plt.title("Hysteresis Area vs Driving Period at Different Temperatures")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# Extended Period Scan at T = 1.5
# ==========================================

EXT_L = 40
EXT_T = 1.5
EXT_h0 = 0.5

# 450 is included to compare with the old plot
EXT_periods = [450, 600, 800, 1000]
EXT_seeds = [101, 202, 303, 404, 505]

EXT_n_cycles = 24
EXT_burn_in = 8

# One result for each independent seed and period
EXT_areas = np.zeros((
    len(EXT_seeds),
    len(EXT_periods)
))

# Preserve cycle-by-cycle data
EXT_cycle_areas = {}

# ==========================================
# Run simulations
# ==========================================

for j, P in enumerate(EXT_periods):

    for i, seed in enumerate(EXT_seeds):

        h, m = simulator(
            L=EXT_L,
            T=EXT_T,
            h0=EXT_h0,
            period=P,
            n=EXT_n_cycles,
            seed=seed
        )

        assert len(h) == EXT_n_cycles * P
        assert len(m) == EXT_n_cycles * P

        # Reshape into separate driving cycles
        h_cycles = np.asarray(h).reshape(
            EXT_n_cycles, P
        )
        m_cycles = np.asarray(m).reshape(
            EXT_n_cycles, P
        )

        # Reuse the ORIGINAL loop_area function
        # to preserve the definition of Area A
        cycle_areas = np.array([
            loop_area(h_cycles[k], m_cycles[k])
            for k in range(EXT_n_cycles)
        ])

        EXT_cycle_areas[(seed, P)] = cycle_areas.copy()

        # Discard transient cycles
        EXT_areas[i, j] = np.mean(
            cycle_areas[EXT_burn_in:]
        )

    print(f"P = {P} completed")


# ==========================================
# Ensemble statistics
# ==========================================

EXT_mean = EXT_areas.mean(axis=0)

EXT_sem = (
    EXT_areas.std(axis=0, ddof=1)
    / np.sqrt(len(EXT_seeds))
)


# ==========================================
# Plot extended results
# ==========================================

fig, ax = plt.subplots(figsize=(9, 5.5))

for i, seed in enumerate(EXT_seeds):
    ax.scatter(
        EXT_periods,
        EXT_areas[i],
        alpha=0.45,
        s=40,
        label=f"Seed {seed}"
    )

ax.errorbar(
    EXT_periods,
    EXT_mean,
    yerr=EXT_sem,
    fmt="ko-",
    linewidth=1.7,
    capsize=5,
    markersize=6,
    label="Mean ± SEM"
)

ax.set_xlabel("Driving Period P (MCS)")
ax.set_ylabel("Hysteresis Loop Area A")

ax.set_title(
    "Extended Period Scan | "
    f"T={EXT_T}, L={EXT_L}, h0={EXT_h0}"
)

ax.grid(alpha=0.25)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


# ==========================================
# Numerical summary
# ==========================================

print("\nExtended Period Scan at T=1.5")
print("-" * 42)

for P, mean, sem in zip(
    EXT_periods, EXT_mean, EXT_sem
):
    print(
        f"P={P:4d} | "
        f"A={mean:.4f} ± {sem:.4f}"
    )


# ==========================================
# Simple burn-in sensitivity check
# ==========================================

print("\nBurn-in sensitivity (8 vs 12 cycles)")

for P in EXT_periods:

    early = []
    late = []

    for seed in EXT_seeds:
        a = EXT_cycle_areas[(seed, P)]

        early.append(np.mean(a[8:]))
        late.append(np.mean(a[12:]))

    print(
        f"P={P:4d} | "
        f"burn=8: {np.mean(early):.4f} | "
        f"burn=12: {np.mean(late):.4f}"
    )



In [ ]:
# ==========================================================
# Multi-Temperature Period Explorer — Part 1
# Precompute Phase-Averaged Hysteresis Loops
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import time

# ----------------------------------------------------------
# 1. Experimental Parameters
# ----------------------------------------------------------

L_TP_demo = 40
h0_TP_demo = 0.5

temperatures_TP_demo = [1.5, 1.8, 2.0, 2.3]

periods_TP_demo = [
    10, 20, 30, 40, 50, 60, 70, 80,
    90, 100, 110, 120, 125, 130, 140, 150,
    175, 200, 225, 250, 275, 300
]

n_cycles_TP_demo = 20
burn_in_TP_demo = 10
seed_TP_demo = 101

# Data storage: data[P][T]
data_TP_demo = {}

start_time = time.perf_counter()

# ----------------------------------------------------------
# 2. Precompute Trajectories
# ----------------------------------------------------------

for P in periods_TP_demo:

    data_TP_demo[P] = {}

    for T in temperatures_TP_demo:

        h, m = simulator(
            L=L_TP_demo,
            T=T,
            h0=h0_TP_demo,
            period=P,
            n=n_cycles_TP_demo,
            seed=seed_TP_demo
        )

        h = np.asarray(h)
        m = np.asarray(m)

        # Split magnetization into separate cycles
        m_cycles = m.reshape(n_cycles_TP_demo, P)

        # Remove initial transient cycles
        m_steady = m_cycles[burn_in_TP_demo:]

        # Average response at each driving phase
        m_mean = np.mean(m_steady, axis=0)

        # One complete field cycle
        h_cycle = h[-P:].copy()

        # Mean signed driving work per spin
        dh = np.roll(h_cycle, -1) - h_cycle
        W = -np.sum(m_mean * dh)

        data_TP_demo[P][T] = {
            "h": h_cycle,
            "m_mean": m_mean,
            "m_raw": m_cycles[-1].copy(),
            "h3": h[-3*P:].copy(),
            "m3": m_cycles[-3:].reshape(-1).copy(),
            "W": W
        }
        

    print(
        f"P = {P:3d} completed | "
        f"Elapsed: {time.perf_counter() - start_time:.1f}s",
        flush=True
    )

print("\nAll T-P combinations completed.")

In [ ]:
# ==========================================================
# Multi-Temperature Period Explorer — Part 2
# Three Cycles + Temperature Checkboxes + Period Playback
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from io import BytesIO
from collections import OrderedDict


# ==========================================================
# 1. Settings
# ==========================================================

periods = [int(P) for P in periods_TP_demo]
temperatures = list(temperatures_TP_demo)

# Consistent colors across both plots
colors = {
    T: plt.cm.tab10(i)
    for i, T in enumerate(temperatures)
}

# Cache rendered images, not Monte Carlo trajectories
image_cache = OrderedDict()
CACHE_LIMIT = 120


# ==========================================================
# 2. Figure Rendering Function
# ==========================================================

def render_TP_figure(P, selected_T):

    selected_T = tuple(selected_T)
    cache_key = (P, selected_T)

    # Reuse an existing image whenever possible
    if cache_key in image_cache:
        image_cache.move_to_end(cache_key)
        return image_cache[cache_key]

    fig, (ax_time, ax_loop) = plt.subplots(
        1, 2,
        figsize=(13, 5)
    )

    # ------------------------------------------------------
    # LEFT: External Field over Three Cycles
    # ------------------------------------------------------

    h3 = np.asarray(
        data_TP_demo[P][temperatures[0]]["h3"]
    )

    t3 = np.arange(3 * P) / P

    ax_time.plot(
        t3,
        h3,
        "k--",
        linewidth=1.6,
        alpha=0.65,
        label="External Field h(t)"
    )

    # ------------------------------------------------------
    # Plot Only Selected Temperatures
    # ------------------------------------------------------

    for T in selected_T:

        data = data_TP_demo[P][T]
        color = colors[T]

        # LEFT: Actual three-cycle response
        m3 = np.asarray(data["m3"])

        ax_time.plot(
            t3,
            m3,
            color=color,
            linewidth=1.7,
            label=f"T = {T:g}"
        )

        # RIGHT: Phase-averaged Hysteresis Loop
        h_cycle = np.asarray(data["h"])
        m_mean = np.asarray(data["m_mean"])

        # Close average loop for visualization
        h_closed = np.r_[h_cycle, h_cycle[0]]
        m_closed = np.r_[m_mean, m_mean[0]]

        ax_loop.plot(
            h_closed,
            m_closed,
            color=color,
            linewidth=2.2,
            label=f"T = {T:g} | W = {data['W']:.2f}"
        )

    # ======================================================
    # 3. LEFT Plot Formatting
    # ======================================================

    ax_time.set_xlim(0, 3)
    ax_time.set_ylim(-1.05, 1.05)

    ax_time.set_xticks([0, 1, 2, 3])

    ax_time.set_xlabel("Time (Driving Cycles)")
    ax_time.set_ylabel("Field h(t) / Magnetization m(t)")

    ax_time.set_title(
        "Magnetization Response — 3 Cycles"
    )

    for boundary in [1, 2]:
        ax_time.axvline(
            boundary,
            color="gray",
            linestyle=":",
            alpha=0.4
        )

    ax_time.axhline(
        0,
        color="gray",
        alpha=0.3
    )

    ax_time.grid(alpha=0.2)
    ax_time.legend(fontsize=9, loc="upper right")

    # ======================================================
    # 4. RIGHT Plot Formatting
    # ======================================================

    ax_loop.set_xlim(
        -1.2 * h0_TP_demo,
        1.2 * h0_TP_demo
    )

    ax_loop.set_ylim(-1.05, 1.05)

    ax_loop.set_xlabel("External Field h")
    ax_loop.set_ylabel("Magnetization m")

    ax_loop.set_title(
        "Hysteresis Loops at Different T"
    )

    ax_loop.axhline(0, color="gray", alpha=0.3)
    ax_loop.axvline(0, color="gray", alpha=0.3)

    ax_loop.grid(alpha=0.2)

    if selected_T:
        ax_loop.legend(
            fontsize=9,
            loc="lower right"
        )
    else:
        ax_loop.text(
            0.5, 0.5,
            "Select a temperature",
            transform=ax_loop.transAxes,
            ha="center",
            va="center",
            fontsize=12,
            color="gray"
        )

    # ======================================================
    # 5. Figure Title
    # ======================================================

    fig.suptitle(
        f"Multi-Temperature Response | "
        f"P = {P} MCS | L = {L_TP_demo}",
        fontsize=14
    )

    fig.tight_layout()

    # ======================================================
    # 6. Save to Memory
    # ======================================================

    buffer = BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=115
    )

    image_bytes = buffer.getvalue()

    buffer.close()
    plt.close(fig)

    # Cache image
    image_cache[cache_key] = image_bytes

    # Limit memory consumption
    if len(image_cache) > CACHE_LIMIT:
        image_cache.popitem(last=False)

    return image_bytes


# ==========================================================
# 7. Temperature Checkboxes
# ==========================================================

temperature_checkboxes = {}

for T in temperatures:

    cb = widgets.Checkbox(
        value=True,
        description=f"T = {T:g}",
        indent=False,
        layout=widgets.Layout(width="105px")
    )

    temperature_checkboxes[T] = cb


def get_selected_temperatures():

    return tuple(
        T for T in temperatures
        if temperature_checkboxes[T].value
    )


# ==========================================================
# 8. Period Slider
# ==========================================================

initial_P = 125 if 125 in periods else periods[0]

period_slider_TP = widgets.SelectionSlider(
    options=[
        (f"P = {P}", P)
        for P in periods
    ],
    value=initial_P,
    description="Period:",
    continuous_update=False,
    layout=widgets.Layout(width="650px")
)


# ==========================================================
# 9. Image Display
# ==========================================================

period_image_TP = widgets.Image(
    value=render_TP_figure(
        initial_P,
        get_selected_temperatures()
    ),
    format="png",
    layout=widgets.Layout(width="100%")
)


# ==========================================================
# 10. Shared Update Function
# ==========================================================

def update_TP_display(change=None):

    P = period_slider_TP.value
    selected_T = get_selected_temperatures()

    period_image_TP.value = render_TP_figure(
        P,
        selected_T
    )


# Slider callback
period_slider_TP.observe(
    update_TP_display,
    names="value"
)

# Checkbox callbacks
for cb in temperature_checkboxes.values():

    cb.observe(
        update_TP_display,
        names="value"
    )


# ==========================================================
# 11. Auto-Play
# ==========================================================

play_TP = widgets.Play(
    value=periods.index(initial_P),
    min=0,
    max=len(periods) - 1,
    step=1,
    interval=450
)

widgets.jslink(
    (play_TP, "value"),
    (period_slider_TP, "index")
)


# ==========================================================
# 12. Optional: Pre-render Selected Configuration
# ==========================================================

preload_button = widgets.Button(
    description="Preload Selected T",
    icon="refresh",
    button_style="",
    tooltip="Cache all periods for the selected temperatures"
)

def preload_selected_T(_):

    selected_T = get_selected_temperatures()

    preload_button.disabled = True

    try:
        for P in periods:
            render_TP_figure(P, selected_T)
    finally:
        preload_button.disabled = False

    update_TP_display()


preload_button.on_click(preload_selected_T)


# ==========================================================
# 13. Display Explorer
# ==========================================================

checkbox_row = widgets.HBox(
    list(temperature_checkboxes.values())
)

control_row = widgets.HBox([
    play_TP,
    period_slider_TP
])

display(
    widgets.VBox([
        widgets.HTML("<b>Temperature Selection</b>"),
        checkbox_row,
        control_row,
        preload_button,
        period_image_TP
    ])
)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from math import ceil

# ==========================================
# 1. Independent experimental parameters
# ==========================================

QPT_L = 40
QPT_h0 = 0.5
QPT_T_values = [1.5, 1.8, 2.0, 2.3]

QPT_base_periods = [
    10, 20, 50, 75, 100,
    125, 150, 200, 300, 450
]

# Additional periods for T = 1.5
QPT_extra_periods = [600, 800, 1000]

QPT_seeds = [101, 202, 303, 404, 505]

# Minimum burn-in and measurement duration
QPT_min_burn_mcs = 2000
QPT_min_measure_mcs = 2000

# Also require sufficient complete cycles
QPT_min_burn_cycles = 8
QPT_min_measure_cycles = 16

# Store summary and per-cycle raw Q
QPT_results = {}
QPT_cycle_data = {}


# ==========================================
# 2. Reuse original simulator()
# ==========================================

for T in QPT_T_values:

    periods = list(QPT_base_periods)

    if T == 1.5:
        periods += QPT_extra_periods

    for P in periods:

        burn_cycles = max(
            QPT_min_burn_cycles,
            ceil(QPT_min_burn_mcs / P)
        )

        measure_cycles = max(
            QPT_min_measure_cycles,
            ceil(QPT_min_measure_mcs / P)
        )

        total_cycles = burn_cycles + measure_cycles

        seed_values = []

        for seed in QPT_seeds:

            # Same initial lattice for a given seed
            rng = np.random.default_rng(100000 + seed)

            initial = rng.choice(
                np.array([-1, 1], dtype=np.int8),
                size=(QPT_L, QPT_L)
            )

            # ORIGINAL simulator
            h, m = simulator(
                L=QPT_L,
                T=T,
                h0=QPT_h0,
                period=P,
                n=total_cycles,
                lattice=initial.copy(),
                seed=seed
            )

            assert len(m) == total_cycles * P

            # Signed Q for each complete cycle
            Q_cycles = np.asarray(m).reshape(
                total_cycles, P
            ).mean(axis=1)

            QPT_cycle_data[(T, P, seed)] = Q_cycles.copy()

            # Discard transient cycles
            Q_retained = Q_cycles[burn_cycles:]

            # Absolute value BEFORE averaging
            seed_values.append(
                np.mean(np.abs(Q_retained))
            )

        seed_values = np.asarray(seed_values)

        QPT_results[(T, P)] = {
            "mean": seed_values.mean(),
            "sem": seed_values.std(ddof=1)
                   / np.sqrt(len(seed_values)),
            "seed_values": seed_values.copy(),
            "burn_cycles": burn_cycles,
            "measure_cycles": measure_cycles
        }

        print(f"T={T:.2f}, P={P:4d} completed")


# ==========================================
# 3. Main Plot: <|Q|> vs P at different T
# ==========================================

fig, ax = plt.subplots(figsize=(10, 6))

for T in QPT_T_values:

    periods = list(QPT_base_periods)

    if T == 1.5:
        periods += QPT_extra_periods

    means = [
        QPT_results[(T, P)]["mean"]
        for P in periods
    ]

    sems = [
        QPT_results[(T, P)]["sem"]
        for P in periods
    ]

    line = ax.errorbar(
        periods,
        means,
        yerr=sems,
        fmt="o-",
        capsize=3,
        linewidth=1.8,
        markersize=5,
        label=f"T = {T}"
    )

    # Individual independent seeds
    for j, P in enumerate(periods):
        values = QPT_results[(T, P)]["seed_values"]

        ax.scatter(
            np.full(len(values), P),
            values,
            color=line[0].get_color(),
            alpha=0.17,
            s=18
        )

ax.set_xlabel("Driving Period P (MCS)")
ax.set_ylabel("Mean |Q|")

ax.set_title(
    "Dynamic Order Parameter vs Driving Period\n"
    f"L={QPT_L}, h0={QPT_h0}"
)

ax.set_ylim(0, 1.05)
ax.set_xlim(left=0)
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


# ==========================================
# 4. Numerical summary
# ==========================================

print("\nTemperature-Period Q Scan")
print("=" * 52)

for T in QPT_T_values:

    print(f"\nT = {T}")

    periods = list(QPT_base_periods)
    if T == 1.5:
        periods += QPT_extra_periods

    for P in periods:
        r = QPT_results[(T, P)]

        print(
            f"P={P:4d} | "
            f"<|Q|> = {r['mean']:.4f} "
            f"± {r['sem']:.4f}"
        )


In [ ]:
# ==========================================================
# STAGE 3 — Finite-Size Analysis
# A(P) and Q(P) for Different Lattice Sizes
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
import time

# ==========================================
# 1. Experimental Parameters
# ==========================================

lattice_sizes_finite = [20, 40, 80]

periods_finite = [
    10, 25, 50, 75, 100,
    125, 150, 200, 300
]

seeds_finite = [101, 202, 303]

T_finite = 2.0
h0_finite = 0.5

n_cycles_finite = 25
burn_in_finite = 10


# ==========================================
# 2. Define Observables
# ==========================================

def signed_work_finite(h, m):
    """
    Signed work per spin over one driving cycle.
    m[t] is measured after the sweep at field h[t].
    """
    h = np.asarray(h)
    m = np.asarray(m)

    dh = np.roll(h, -1) - h

    return -np.sum(m * dh)


# Shape: (L, P, seed)
shape_finite = (
    len(lattice_sizes_finite),
    len(periods_finite),
    len(seeds_finite)
)

all_W_finite = np.zeros(shape_finite)
all_Aabs_finite = np.zeros(shape_finite)
all_Q_finite = np.zeros(shape_finite)


# ==========================================
# 3. Run Monte Carlo Simulations
# ==========================================

start_time = time.perf_counter()

for i, L in enumerate(lattice_sizes_finite):

    print(f"\nRunning L = {L}", flush=True)

    for j, P in enumerate(periods_finite):

        for s, seed in enumerate(seeds_finite):

            h, m = simulator(
                L=L,
                T=T_finite,
                h0=h0_finite,
                period=P,
                n=n_cycles_finite,
                seed=seed
            )

            h = np.asarray(h)
            m = np.asarray(m)

            # Divide data into individual cycles
            h_cycles = h.reshape(n_cycles_finite, P)
            m_cycles = m.reshape(n_cycles_finite, P)

            # Discard transient cycles
            h_steady = h_cycles[burn_in_finite:]
            m_steady = m_cycles[burn_in_finite:]

            # ------ Hysteresis Work ------

            cycle_W = []

            for k in range(len(h_steady)):

                W = signed_work_finite(
                    h_steady[k],
                    m_steady[k]
                )

                cycle_W.append(W)

            cycle_W = np.asarray(cycle_W)

            # Signed mean work
            all_W_finite[i, j, s] = np.mean(cycle_W)

            # Old definition: mean absolute area
            all_Aabs_finite[i, j, s] = np.mean(
                np.abs(cycle_W)
            )

            # ------ Dynamic Order Parameter ------

            Q_cycles = np.mean(m_steady, axis=1)

            all_Q_finite[i, j, s] = np.mean(
                np.abs(Q_cycles)
            )

        print(f"  P = {P:3d} completed", flush=True)


print(
    f"\nCompleted in "
    f"{time.perf_counter() - start_time:.2f} seconds"
)


# ==========================================
# 4. Ensemble Statistics
# ==========================================

def ensemble_statistics_finite(data):

    mean = np.mean(data, axis=2)

    sem = (
        np.std(data, axis=2, ddof=1)
        / np.sqrt(data.shape[2])
    )

    return mean, sem


mean_W_finite, error_W_finite = (
    ensemble_statistics_finite(all_W_finite)
)

mean_Aabs_finite, error_Aabs_finite = (
    ensemble_statistics_finite(all_Aabs_finite)
)

mean_Q_finite, error_Q_finite = (
    ensemble_statistics_finite(all_Q_finite)
)


# ==========================================
# 5. Final Plot
# ==========================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5)
)

# ------ Left: Mean Dissipated Work ------

for i, L in enumerate(lattice_sizes_finite):

    axes[0].errorbar(
        periods_finite,
        mean_W_finite[i],
        yerr=error_W_finite[i],
        fmt='o-',
        capsize=3,
        label=f'L = {L}'
    )

axes[0].set_xlabel("Driving Period P (MCS)")
axes[0].set_ylabel("Mean Signed Work per Spin")
axes[0].set_title("Finite-Size Dependence of Hysteresis")
axes[0].legend()
axes[0].grid(alpha=0.3)


# ------ Right: Dynamic Order Parameter ------

for i, L in enumerate(lattice_sizes_finite):

    axes[1].errorbar(
        periods_finite,
        mean_Q_finite[i],
        yerr=error_Q_finite[i],
        fmt='o-',
        capsize=3,
        label=f'L = {L}'
    )

axes[1].set_xlabel("Driving Period P (MCS)")
axes[1].set_ylabel("Mean |Q|")
axes[1].set_title("Finite-Size Dependence of Dynamic Order")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle(
    f"Finite-Size Analysis | T = {T_finite}, "
    f"h0 = {h0_finite}",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

# ============================================================
# Finite-Size Reliability Check
# Uses the ORIGINAL simulator() without modifying its physics
# ============================================================

LATTICE_SIZES = [20, 40, 80]
PERIODS_TO_CHECK = [50]  # Optional: [50, 125]
INITIAL_STATES = ["all-up", "all-down", "random"]
SEEDS = [101, 202, 303, 404, 505, 606, 707, 808]

T_CHECK = 2.0
H0_CHECK = 0.5

BURN_MCS = 3000
MEASURE_MCS = 6000


def initial_lattice(L, state):
    if state == "all-up":
        return np.ones((L, L), dtype=np.int8)
    if state == "all-down":
        return -np.ones((L, L), dtype=np.int8)
    if state == "random":
        return None  # Original simulator generates the lattice
    raise ValueError(f"Unknown state: {state}")


def observables_by_cycle(h, m, P):
    h = np.asarray(h).reshape(-1, P)
    m = np.asarray(m).reshape(-1, P)

    dh = np.roll(h, -1, axis=1) - h

    # Signed cycle-averaged magnetization
    q = m.mean(axis=1)

    # Signed discrete work, consistent with simulator timing
    w = -np.sum(m * dh, axis=1)

    # Original absolute trapezoidal integral
    m_trap = 0.5 * (m + np.roll(m, -1, axis=1))
    a_trap = np.abs(-np.sum(m_trap * dh, axis=1))

    return q, w, a_trap


def mean_ci(values):
    values = np.asarray(values, dtype=float)
    n = len(values)
    mean = values.mean()

    if n < 2:
        return mean, np.nan

    sem = values.std(ddof=1) / np.sqrt(n)
    half_width = student_t.ppf(0.975, n - 1) * sem
    return mean, half_width


results = []
traces = {}

for P in PERIODS_TO_CHECK:

    # Same approximate number of MCS for every tested P
    burn_cycles = int(np.ceil(BURN_MCS / P))
    measure_cycles = int(np.ceil(MEASURE_MCS / P))
    total_cycles = burn_cycles + measure_cycles

    print(
        f"\nP={P}: burn={burn_cycles * P} MCS, "
        f"measure={measure_cycles * P} MCS",
        flush=True
    )

    for L in LATTICE_SIZES:
        for state in INITIAL_STATES:
            for seed in SEEDS:

                h, m = simulator(
                    L=L,
                    T=T_CHECK,
                    h0=H0_CHECK,
                    period=P,
                    n=total_cycles,
                    lattice=initial_lattice(L, state),
                    seed=seed
                )

                q, w, a_trap = observables_by_cycle(h, m, P)

                # Keep only post-burn-in cycles
                q = q[burn_cycles:]
                w = w[burn_cycles:]
                a_trap = a_trap[burn_cycles:]

                # Divide retained cycles into two windows
                mid = len(q) // 2

                q_first = np.abs(q[:mid]).mean()
                q_second = np.abs(q[mid:]).mean()

                w_first = w[:mid].mean()
                w_second = w[mid:].mean()

                results.append({
                    "L": L,
                    "P": P,
                    "state": state,
                    "seed": seed,
                    "Qabs": np.abs(q).mean(),
                    "Qsigned": q.mean(),
                    "W": w.mean(),
                    "Atrap": a_trap.mean(),
                    "drift_Q": q_second - q_first,
                    "drift_W": w_second - w_first
                })

                # Save one representative signed-Q trajectory
                if seed == SEEDS[0]:
                    times = (
                        np.arange(len(q)) + burn_cycles
                    ) * P
                    traces[(L, P, state)] = (times, q)

            print(f"Finished L={L}, P={P}, {state}", flush=True)


def group_values(L, P, state, key):
    return [
        row[key] for row in results
        if row["L"] == L
        and row["P"] == P
        and row["state"] == state
    ]


# ============================================================
# Summary tables and plots
# ============================================================

for P in PERIODS_TO_CHECK:

    print(f"\n{'=' * 70}")
    print(f"Finite-size reliability results | P={P}")
    print("Intervals: approximate 95% CI across independent seeds")
    print(f"{'=' * 70}")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    keys = ["Qabs", "W", "drift_Q"]
    titles = [
        "Mean Absolute Dynamic Order",
        "Mean Signed Work per Spin",
        "Q: Late Window Minus Early Window"
    ]
    ylabels = ["Mean |Q|", "Mean Signed W", "Change in Mean |Q|"]

    for state in INITIAL_STATES:

        for ax, key in zip(axes, keys):
            means, errors = [], []

            for L in LATTICE_SIZES:
                mu, ci = mean_ci(
                    group_values(L, P, state, key)
                )
                means.append(mu)
                errors.append(ci)

            ax.errorbar(
                LATTICE_SIZES, means, yerr=errors,
                marker="o", capsize=4, label=state
            )

    for ax, title, ylabel in zip(axes, titles, ylabels):
        ax.set_title(title)
        ax.set_xlabel("Lattice size L")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
        ax.legend()

    axes[2].axhline(0, color="gray", ls="--")

    fig.suptitle(
        f"Finite-Size Reliability | T={T_CHECK}, "
        f"P={P}, h0={H0_CHECK}"
    )

    plt.tight_layout()
    plt.show()

    # Numerical summary for each initial-state group
    for L in LATTICE_SIZES:
        for state in INITIAL_STATES:

            q_mean, q_ci = mean_ci(
                group_values(L, P, state, "Qabs")
            )
            w_mean, w_ci = mean_ci(
                group_values(L, P, state, "W")
            )
            d_mean, d_ci = mean_ci(
                group_values(L, P, state, "drift_Q")
            )

            print(
                f"L={L:3d}, {state:8s} | "
                f"|Q|={q_mean:.4f} +/- {q_ci:.4f} | "
                f"W={w_mean:.4f} +/- {w_ci:.4f} | "
                f"drift_Q={d_mean:+.4f} +/- {d_ci:.4f}"
            )


# ============================================================
# Additional diagnostic: signed Q trajectories at largest L
# ============================================================

P = PERIODS_TO_CHECK[0]
L = max(LATTICE_SIZES)

plt.figure(figsize=(10, 4.5))

for state in INITIAL_STATES:
    times, q = traces[(L, P, state)]
    plt.plot(times, q, label=state, alpha=0.85)

plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Elapsed MCS")
plt.ylabel("Signed Q per Cycle")
plt.title(
    f"Post-Burn-in Magnetization Dynamics | L={L}, P={P}"
)
plt.legend()
plt.grid(alpha=0.3)
plt.show()
